<a href="https://colab.research.google.com/github/minseonju/Marine/blob/main/model/lstm/LSTM_ae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# [셀 1] Google Drive 마운트
# 학습 데이터(raw_events.csv, raw_attack/*.csv)가 Drive에 저장되어 있으므로
# 가장 먼저 Drive를 마운트해야 파일 경로에 접근할 수 있음
# ================================================================
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ================================================================
# [셀 2] 라이브러리 import & 전역 설정
# - random seed 고정 : 재현성 보장
# - 하이퍼파라미터를 config에서 관리
# - 스케일러를 딕셔너리로 등록
# ================================================================
import os
import pickle
import random

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
    classification_report,
)

# ── 재현성 고정 ──────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── 하이퍼파라미터 ───────────────────────────────────────────────────────────
config = {
    'base_path'      : None,
    'seq_len'        : 100,    # 슬라이딩 윈도우 시퀀스 길이
    'stride'         : 10,     # 윈도우 이동 간격
    'batch_size'     : 256,    # 미니배치 크기
    'epochs'         : 70,     # 전체 학습 반복 횟수
    'hidden_dim'     : 128,    # LSTM 은닉 상태 차원
    'lr'             : 0.0005, # Adam 초기 학습률
    'weight_decay'   : 1e-5,   # L2 정규화 계수
    'scheduler_step' : 30,     # StepLR 주기 (epoch 단위)
    'scheduler_gamma': 0.5,    # StepLR 감소 비율
    'scaler'         : 'standard',  # 'standard' | 'minmax' | 'robust'
    'normal_sample_n': 200_000,     # 학습에 사용할 정상 시퀀스 수
}

SCALERS = {
    'standard': StandardScaler(),
    'minmax'  : MinMaxScaler(),
    'robust'  : RobustScaler(),
}
scaler = SCALERS[config['scaler']]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEQ_LEN = config['seq_len']
STRIDE  = config['stride']

print(f"사용 디바이스: {device}")
print(f"스케일러: {config['scaler']}")


In [ ]:
# ================================================================
# [셀 3] 원시 데이터 로드 & syscall 시퀀스 생성
# ================================================================
import pandas as pd

# ── 정상 데이터 ──────────────────────────────────────────────────────────────
df_normal = pd.read_csv('/content/drive/MyDrive/raw_events.csv')
df_normal = df_normal.sort_values('ts_ns').reset_index(drop=True)
print(f"정상 원본: {df_normal.shape}")

# syscall 문자열 → 정수 인덱스 매핑 (이후 공격 데이터도 같은 사전 사용)
unique_syscalls = sorted(df_normal['syscall'].unique())
syscall_to_idx  = {s: i for i, s in enumerate(unique_syscalls)}
df_normal['syscall_idx'] = df_normal['syscall'].map(syscall_to_idx)
print(f"고유 syscall 수: {len(unique_syscalls)}개")

# 슬라이딩 윈도우로 정상 시퀀스 생성
normal_seqs: list[list[int]] = []
seq_arr = df_normal['syscall_idx'].values
for i in range(0, len(seq_arr) - SEQ_LEN + 1, STRIDE):
    normal_seqs.append(seq_arr[i:i + SEQ_LEN].tolist())
print(f"정상 시퀀스 총합: {len(normal_seqs):,}개")

# ── 공격 데이터 ──────────────────────────────────────────────────────────────
ATTACK_FILES = {
    'rs_bash_post' : '/content/drive/MyDrive/raw_attack/raw_rs_bash_post.csv',
    'rs_socat'     : '/content/drive/MyDrive/raw_attack/raw_rs_socat.csv',
    'rs_perl_post' : '/content/drive/MyDrive/raw_attack/raw_rs_perl_post.csv',
    'rs_ruby'      : '/content/drive/MyDrive/raw_attack/raw_rs_ruby.csv',
    'rs_python3'   : '/content/drive/MyDrive/raw_attack/raw_rs_python3.csv',
    'rs_ncmkfifo'  : '/content/drive/MyDrive/raw_attack/raw_rs_ncmkfifo.csv',
    'rs_php'       : '/content/drive/MyDrive/raw_attack/raw_rs_php.csv',
    'crypto_mining': '/content/drive/MyDrive/raw_attack/raw_crypto_mining.csv',
}

attack_seqs  : list[list[int]] = []
attack_labels: list[str]       = []

for label, fpath in ATTACK_FILES.items():
    df_atk = pd.read_csv(fpath).sort_values('ts_ns').reset_index(drop=True)
    df_atk['syscall_idx'] = (
        df_atk['syscall'].map(syscall_to_idx).fillna(0).astype(int)
    )
    seqs = df_atk['syscall_idx'].values
    count = 0
    for i in range(0, len(seqs) - SEQ_LEN + 1, STRIDE):
        attack_seqs.append(seqs[i:i + SEQ_LEN].tolist())
        attack_labels.append(label)
        count += 1
    print(f"  {label:20s}: {count:,}개 시퀀스")

print(f"\n공격 시퀀스 총합: {len(attack_seqs):,}개")

# ── 샘플링 & 테스트 분할 ─────────────────────────────────────────────────────
random.seed(SEED)
normal_seqs      = random.sample(normal_seqs, config['normal_sample_n'])
n_attack         = len(attack_seqs)
normal_test_seqs = random.sample(normal_seqs, n_attack)  # 1:1 비율

print(f"\n=== 데이터 구성 ===")
print(f"학습용 정상 : {len(normal_seqs):,}개")
print(f"테스트 정상 : {len(normal_test_seqs):,}개")
print(f"테스트 공격 : {n_attack:,}개  (비율 1:1)")


In [ ]:
# ================================================================
# [셀 4] 스케일링 & PyTorch 텐서 변환
# ================================================================
normal_arr      = np.array(normal_seqs,      dtype=float)
normal_test_arr = np.array(normal_test_seqs, dtype=float)
attack_arr      = np.array(attack_seqs,      dtype=float)

normal_scaled      = scaler.fit_transform(normal_arr)       # fit + transform
normal_test_scaled = scaler.transform(normal_test_arr)      # transform only
attack_scaled      = scaler.transform(attack_arr)           # transform only

X_train       = torch.tensor(normal_scaled,      dtype=torch.float32).unsqueeze(-1).to(device)
X_normal_test = torch.tensor(normal_test_scaled, dtype=torch.float32).unsqueeze(-1).to(device)
X_attack      = torch.tensor(attack_scaled,      dtype=torch.float32).unsqueeze(-1).to(device)

print(f"학습 텐서       : {X_train.shape}")
print(f"테스트 정상 텐서: {X_normal_test.shape}")
print(f"테스트 공격 텐서: {X_attack.shape}")


In [ ]:
# ================================================================
# [셀 5] LSTM-AE 모델 정의 & 학습
# 모델 구조:
#   Encoder: LSTM (n_features → hidden_dim, 2 layer, dropout=0.2)
#   Decoder: LSTM (hidden_dim → n_features, 2 layer, dropout=0.2)
#   forward : 인코더 마지막 hidden state를 seq_len 번 반복해
#             디코더 입력으로 사용 → 시퀀스를 복원
# 학습 전략:
#   - Loss     : MSELoss (복원 오차 최소화)
#   - Optimizer: Adam + weight_decay (L2 정규화)
#   - Scheduler: StepLR (주기적 학습률 감소)
#   - Grad clip: clip_grad_norm_(1.0) (기울기 폭발 방지)
#   - DataLoader shuffle=True (매 epoch 배치 순서 섞기)
# ================================================================
from torch.utils.data import DataLoader, TensorDataset


class LSTM_AE(nn.Module):
    def __init__(
        self,
        seq_len   : int = SEQ_LEN,
        n_features: int = 1,
        hidden_dim: int = config['hidden_dim'],
    ) -> None:
        super().__init__()
        self.seq_len = seq_len
        self.encoder = nn.LSTM(
            n_features, hidden_dim,
            num_layers=2, batch_first=True, dropout=0.2,
        )
        self.decoder = nn.LSTM(
            hidden_dim, n_features,
            num_layers=2, batch_first=True, dropout=0.2,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, (hidden, _) = self.encoder(x)
        # 마지막 hidden state를 seq_len 번 반복해 디코더 초기 입력으로 사용
        repeated = hidden[-1].unsqueeze(0).repeat(self.seq_len, 1, 1).transpose(0, 1)
        x_rec, _ = self.decoder(repeated)
        return x_rec


# ── 모델 / 옵티마이저 / 스케줄러 초기화 ─────────────────────────────────────
model     = LSTM_AE().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config['lr'],
    weight_decay=config['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=config['scheduler_step'],
    gamma=config['scheduler_gamma'],
)

loader = DataLoader(
    TensorDataset(X_train),
    batch_size=config['batch_size'],
    shuffle=True,
)

# ── 학습 루프 ────────────────────────────────────────────────────────────────
train_losses: list[float] = []

print("학습 시작...")
for epoch in range(config['epochs']):
    model.train()
    total_loss = 0.0

    for (batch,) in loader:
        optimizer.zero_grad()
        output = model(batch)
        loss   = criterion(output, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(loader)
    train_losses.append(avg_loss)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Loss: {avg_loss:.6f} | LR: {scheduler.get_last_lr()[0]:.6f}")

print("학습 완료!")


In [ ]:
# ================================================================
# [셀 6] 복원 오차 계산 & 이상 탐지 임계값 결정
# 임계값 탐색:
#   정상 오차 분위수(percentile)를 0.5 간격으로 순회하며
#   macro F1이 최대가 되는 임계값 선택
# ================================================================

def compute_reconstruction_errors(
    model   : nn.Module,
    X       : torch.Tensor,
    batch_sz: int,
) -> np.ndarray:
    """모델의 복원 오차(MSE)를 배치 단위로 계산해 반환한다."""
    model.eval()
    errors: list[float] = []
    with torch.no_grad():
        for i in range(0, len(X), batch_sz):
            batch = X[i:i + batch_sz]
            pred  = model(batch)
            err   = torch.mean((pred - batch) ** 2, dim=(1, 2))
            errors.extend(err.cpu().numpy())
    return np.array(errors)


BATCH_SIZE    = config['batch_size']
normal_errors = compute_reconstruction_errors(model, X_normal_test, BATCH_SIZE)
attack_errors = compute_reconstruction_errors(model, X_attack,      BATCH_SIZE)

print(f"정상 오차 — mean: {normal_errors.mean():.6f}, std: {normal_errors.std():.6f}")
print(f"공격 오차 — mean: {attack_errors.mean():.6f}, std: {attack_errors.std():.6f}")

# ── 반전 여부 확인 ───────────────────────────────────────────────────────────
all_errors = np.concatenate([normal_errors, attack_errors])
invert     = attack_errors.mean() < normal_errors.mean()
if invert:
    print("\n공격 오차가 정상보다 낮음 → 이상 점수 반전 적용")
else:
    print("\n정상적인 방향 (공격 오차 > 정상 오차)")

# ── 임계값 탐색 (macro F1 최대화) ───────────────────────────────────────────
y_true = np.array([0] * len(normal_errors) + [1] * len(attack_errors))
pct_range = np.arange(1, 50, 0.5) if invert else np.arange(50, 99, 0.5)

best_f1, best_t = 0.0, 0.0
for pct in pct_range:
    t    = np.percentile(normal_errors, pct)
    pred = (all_errors < t).astype(int) if invert else (all_errors > t).astype(int)
    f1   = f1_score(y_true, pred, average='macro')
    if f1 > best_f1:
        best_f1, best_t = f1, t

threshold = best_t
y_pred    = (all_errors < threshold).astype(int) if invert else (all_errors > threshold).astype(int)
print(f"\n최적 임계값: {threshold:.6f}  (macro F1: {best_f1:.4f})")

# ── ROC & 혼동행렬 ───────────────────────────────────────────────────────────
score_for_roc            = -all_errors if invert else all_errors
fpr_arr, tpr_arr, _      = roc_curve(y_true, score_for_roc)
tn, fp, fn, tp           = confusion_matrix(y_true, y_pred).ravel()

print(f"\n=== 성능 평가 ===")
print(f"TP: {tp}  TN: {tn}  FP: {fp}  FN: {fn}")
print(f"Precision : {precision_score(y_true, y_pred):.4f}")
print(f"Recall    : {recall_score(y_true, y_pred):.4f}")
print(f"F1-Score  : {f1_score(y_true, y_pred):.4f}")
print(f"FPR       : {fp / (fp + tn):.4f}")
print(f"AUC-ROC   : {roc_auc_score(y_true, score_for_roc):.4f}")

# ── 공격 유형별 탐지율 ───────────────────────────────────────────────────────
attack_label_arr = np.array(attack_labels)
detection_rates  : dict[str, float] = {}

print("\n[공격 유형별 탐지율]")
for atype in sorted(set(attack_labels)):
    mask     = attack_label_arr == atype
    detected = (
        np.sum(attack_errors[mask] < threshold) if invert
        else np.sum(attack_errors[mask] > threshold)
    )
    total    = int(np.sum(mask))
    rate     = detected / total * 100
    detection_rates[atype] = rate
    print(f"  {atype:20s}: {detected}/{total} ({rate:.1f}%)")


In [ ]:
# ================================================================
# [셀 7] 결과 시각화 (2개 figure, 총 7개 서브플롯)
# Figure 1 (4개):
#   ① Training Loss 곡선
#   ② 공격 유형별 복원 오차 KDE 분포
#   ③ ROC Curve
#   ④ Percentile별 macro F1 (임계값 탐색 결과)
# Figure 2 (3개):
#   ⑤ 공격 유형별 Anomaly Score 히스토그램
#   ⑥ 혼동행렬 히트맵
#   ⑦ 공격 유형별 탐지율 막대그래프
# ================================================================
import seaborn as sns
from scipy.stats import gaussian_kde

# ── 한글 폰트 설정 (Colab 전용) ─────────────────────────────────────────────
!apt-get install -y fonts-nanum -qq
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family']        = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

COLORS     = ['tomato', 'orange', 'green', 'purple', 'brown', 'pink', 'gray']
atypes_srt = sorted(set(attack_labels))
x_range    = np.linspace(0, max(normal_errors.max(), attack_errors.max()), 300)

# ── Figure 1 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('LSTM-AE — Loss / Error Distribution / ROC / Threshold Search')

# ① Training Loss
axes[0].plot(train_losses, color='steelblue')
axes[0].set(title='Training Loss', xlabel='Epoch', ylabel='MSE Loss')
axes[0].grid(True, alpha=0.3)

# ② 복원 오차 KDE 분포
kde_normal = gaussian_kde(normal_errors)
axes[1].plot(x_range, kde_normal(x_range), color='steelblue',
             label=f'Normal (n={len(normal_errors)})')
for idx, atype in enumerate(atypes_srt):
    mask = attack_label_arr == atype
    errs = attack_errors[mask]
    if len(errs) > 10:
        axes[1].plot(x_range, gaussian_kde(errs)(x_range),
                     label=f'{atype} (n={len(errs)})',
                     color=COLORS[idx % len(COLORS)], alpha=0.7)
axes[1].axvline(threshold, color='black', linestyle='--',
                label=f'Threshold={threshold:.3f}')
axes[1].set(title='Reconstruction Error Distribution',
            xlabel='MSE Error', ylabel='Density')
axes[1].legend(fontsize=7)

# ③ ROC Curve
axes[2].plot(fpr_arr, tpr_arr, color='blue',
             label=f'AUC={roc_auc_score(y_true, score_for_roc):.4f}')
axes[2].plot([0, 1], [0, 1], 'k--', label='Random')
axes[2].set(title='ROC Curve', xlabel='FPR', ylabel='TPR')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# ④ Threshold Search (macro F1)
f1_list = []
for pct in pct_range:
    t    = np.percentile(normal_errors, pct)
    pred = (all_errors < t).astype(int) if invert else (all_errors > t).astype(int)
    f1_list.append(f1_score(y_true, pred, average='macro'))
best_pct = pct_range[int(np.argmax(f1_list))]
axes[3].plot(pct_range, f1_list, color='steelblue')
axes[3].axvline(best_pct, color='red', linestyle='--',
                label=f'Best={best_pct:.1f}th\nF1={max(f1_list):.4f}')
axes[3].set(title='Threshold Search (macro F1)',
            xlabel='Percentile', ylabel='macro F1')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lstm_raw_result.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2 ────────────────────────────────────────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle('LSTM-AE — Score Distribution / Confusion Matrix / Detection Rate')

# ⑤ Anomaly Score 히스토그램
axes2[0].hist(normal_errors, bins=60, alpha=0.5, density=True,
              color='steelblue', label=f'Normal ({len(normal_errors)})')
for idx, atype in enumerate(atypes_srt):
    mask = attack_label_arr == atype
    axes2[0].hist(attack_errors[mask], bins=60, alpha=0.5, density=True,
                  label=f'{atype} ({mask.sum()})',
                  color=COLORS[idx % len(COLORS)])
axes2[0].axvline(threshold, color='black', linestyle='--',
                 label=f'Threshold={threshold:.3f}')
axes2[0].set(title='Anomaly Score Distribution',
             xlabel='MSE Error', ylabel='Density')
axes2[0].legend(fontsize=7)

# ⑥ 혼동행렬 히트맵
sns.heatmap(
    confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred Normal', 'Pred Attack'],
    yticklabels=['Actual Normal', 'Actual Attack'],
    ax=axes2[1],
)
axes2[1].set_title('Confusion Matrix')

# ⑦ 공격 유형별 탐지율 막대그래프
rates = [detection_rates[a] for a in atypes_srt]
bars  = axes2[2].bar(atypes_srt, rates, color='steelblue', alpha=0.8)
axes2[2].axhline(y=80, color='red', linestyle='--', label='80% 기준선')
axes2[2].set(title='Attack Detection Rate by Type',
             ylabel='Detection Rate (%)', ylim=(0, 110))
axes2[2].tick_params(axis='x', rotation=45)
axes2[2].legend()
for bar, rate in zip(bars, rates):
    axes2[2].text(
        bar.get_x() + bar.get_width() / 2.,
        bar.get_height() + 1,
        f'{rate:.1f}%', ha='center', va='bottom', fontsize=9,
    )

plt.tight_layout()
plt.savefig('lstm_raw_result2.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Classification Report ────────────────────────────────────────────────────
print("=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Attack']))
print(f"ROC-AUC : {roc_auc_score(y_true, score_for_roc):.4f}")
print(f"Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")


In [ ]:
# ================================================================
# [셀 8] 모델 & 앙상블용 파일 저장
# 로컬 저장 (model/ 디렉터리):
#   - lstm_ae.pth          : 학습된 가중치 (추론 시 로드)
#   - scaler.pkl           : 학습 데이터 기준 스케일러 (새 데이터 전처리에 사용)
#   - threshold.npy        : 최적 임계값
#   - normal_err_stats.npy : 정상 오차 통계 (mean/std/min/max, 모니터링용)
# Google Drive 저장 (앙상블 모델과 결과 공유용):
#   - lstm_normal_errors.npy / lstm_attack_errors.npy
#   - lstm_invert.npy / lstm_threshold.npy / lstm_detection_rates.npy
# ================================================================
os.makedirs('model', exist_ok=True)

# ── 로컬 저장 ────────────────────────────────────────────────────────────────
torch.save(model.state_dict(), 'model/lstm_ae.pth')
with open('model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
np.save('model/threshold.npy', threshold)
np.save('model/normal_err_stats.npy',
        np.array([normal_errors.mean(), normal_errors.std(),
                  normal_errors.min(), normal_errors.max()]))

print("로컬 저장 완료: model/lstm_ae.pth, scaler.pkl, threshold.npy, normal_err_stats.npy")

# ── Google Drive 저장 (앙상블용) ─────────────────────────────────────────────
DRIVE_OUT = '/content/drive/MyDrive'
np.save(f'{DRIVE_OUT}/lstm_normal_errors.npy',   normal_errors)
np.save(f'{DRIVE_OUT}/lstm_attack_errors.npy',   attack_errors)
np.save(f'{DRIVE_OUT}/lstm_invert.npy',          np.array(invert))
np.save(f'{DRIVE_OUT}/lstm_threshold.npy',       threshold)
np.save(f'{DRIVE_OUT}/lstm_detection_rates.npy', detection_rates)

print("Drive 저장 완료: lstm_*.npy")
